# ⚔️ B5　Boss 戰：銷售報表產生器
**Python 冒險之旅 2026**　｜　Day 5（09/04 五）🏜️ 檔案之島　｜　Boss 戰　｜　🏅 200 XP

📖 對應教科書：第 9–10 章綜合


### 🎯 這一關你會學到
- 讀檔 → 例外處理 → 統計 → 畫圖的完整資料流程

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "B5"
_SALT = "python-quest-2026-datama"
_TASKS = ["B5-1", "B5-2", "B5-3", "B5-4", "B5-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B5_1(run):
    out, ns = run()
    r = ns.get("records", [])
    if len(r) != 7: return (False, f"有效資料應該 7 筆，現在 {len(r)} 筆（壞資料有 2 行）。")
    if out.count("略過壞資料") != 2: return (False, "應該印出 2 次 略過壞資料。")
    return (r[0] == ('珍珠奶茶', 12, 60), "每筆應該是 (品名, 整數數量, 整數單價)。")
任務定義("B5-1", _check_B5_1, 提示="except ValueError: 就能接住轉型與拆解失敗（兩者都是 ValueError）。")

def _check_B5_2(run):
    out, ns = run()
    want = {'珍珠奶茶': 1020, '雞排': 600, '滷肉飯': 990, '鹹酥雞': 480, '紅茶': 600}
    return (ns.get("revenue") == want, f"revenue 應該是 {want}，現在是 {ns.get('revenue')}。")
任務定義("B5-2", _check_B5_2, 提示="revenue[name] = revenue.get(name, 0) + qty * price。")

def _check_B5_3(run):
    out, ns = run()
    lines = 行列表(out)
    if not (lines[0].startswith("第1名 珍珠奶茶") and "1,020" in lines[0]): return (False, "第 1 名應該是珍珠奶茶 1,020 元。")
    if not lines[1].startswith("第2名 滷肉飯"): return (False, "第 2 名應該是滷肉飯。")
    return (出現(out, "總營收3,690元"), "總營收應該是 3,690 元（用千分位）。")
任務定義("B5-3", _check_B5_3, 提示="{money:,} 千分位；sum(revenue.values()) 是總和。")

def _check_B5_4(run):
    import os
    if os.path.exists('revenue.png'): os.remove('revenue.png')
    out, ns = run()
    import matplotlib.pyplot as plt
    ax = plt.gcf().axes[0] if plt.gcf().axes else None
    if ax is None or len(ax.patches) != 5: return (False, "應該有 5 根柱子。")
    if sorted(p.get_height() for p in ax.patches) != [480, 600, 600, 990, 1020]: return (False, "柱子高度要是各品項營收。")
    ok = ax.get_title() == '品項營收' and ax.get_ylabel() == '營收（元）'
    plt.close('all')
    if not ok: return (False, "標題 品項營收、y 軸 營收（元）。")
    return (os.path.exists('revenue.png'), "要 savefig('revenue.png')。")
任務定義("B5-4", _check_B5_4, 提示="values = list(revenue.values())。")

def _check_B5_5(run):
    out, ns = run()
    f = ns.get("make_report")
    if not callable(f): return (False, "要定義 make_report。")
    want = {'珍珠奶茶': 1020, '雞排': 600, '滷肉飯': 990, '鹹酥雞': 480, '紅茶': 600}
    return (ns.get("result") == want and 出現(out, "總營收3,690元"), "回傳的字典與總營收不對。")
任務定義("B5-5", _check_B5_5, 提示="revenue[name] = revenue.get(name, 0) + int(qty) * int(price)。")


## ⚔️ Boss 登場：銷售報表產生器
老闆丟給你一個銷售紀錄檔 `sales.txt`（每行：`品名,數量,單價`），裡面**有幾行壞掉的資料**。
你要：讀檔 → 略過壞資料（例外處理）→ 統計每個品項的營收 → 排序 → 畫圖存檔。

先執行下面這格產生資料檔：

In [ ]:
raw = """珍珠奶茶,12,60
雞排,8,75
滷肉飯,15,45
紅茶,abc,30
鹹酥雞,6,80
珍珠奶茶,5,60
雞排,3
紅茶,20,30
滷肉飯,7,45
"""
with open('sales.txt', 'w', encoding='utf-8') as f:
    f.write(raw)
print(open('sales.txt', encoding='utf-8').read())

In [ ]:
#@title 🈶 中文字型設定（Colab 預設沒有中文字型，執行這一格安裝；約 20～40 秒）
import subprocess, glob, matplotlib
from matplotlib import font_manager
subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1", shell=True)
for f in glob.glob('/usr/share/fonts/opentype/noto/NotoSansCJK*-Regular.ttc'):
    font_manager.fontManager.addfont(f)
matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'    # 這個字型檔同時包含繁體中文字形
matplotlib.rcParams['axes.unicode_minus'] = False          # 讓負號正常顯示
print("✅ 中文字型設定完成（若圖表中文仍是方框，請重新執行這一格後再畫一次）")

### 🎯 任務 B5-1　安全讀檔

讀取 `sales.txt`，把每一行解析成 `(品名, 數量, 單價)` 加入串列 `records`；解析失敗的行（欄位不足或不是數字）用 `try/except` 略過並印出 `略過壞資料：...`。最後印出 `有效資料 7 筆`。

In [ ]:
# 🎯 任務 B5-1　安全讀檔（請保留這一行）
records = []
with open('sales.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            name, qty, price = line.split(',')
            records.append((name, int(qty), int(price)))
        except ???:
            print('略過壞資料：', line)
print(f"有效資料 {len(records)} 筆")

In [ ]:
檢查("B5-1")   # ◀ 執行這一格，看看任務 B5-1 有沒有過關

### 🎯 任務 B5-2　品項營收統計

用字典 `revenue` 累計每個品項的營收（數量 × 單價）。預期：`{'珍珠奶茶': 1020, '雞排': 600, '滷肉飯': 990, '鹹酥雞': 480, '紅茶': 600}`（順序不拘）。

In [ ]:
# 🎯 任務 B5-2　品項營收統計（請保留這一行）
records = [('珍珠奶茶', 12, 60), ('雞排', 8, 75), ('滷肉飯', 15, 45), ('鹹酥雞', 6, 80),
           ('珍珠奶茶', 5, 60), ('紅茶', 20, 30), ('滷肉飯', 7, 45)]
revenue = {}
for name, qty, price in records:
    ???
print(revenue)

In [ ]:
檢查("B5-2")   # ◀ 執行這一格，看看任務 B5-2 有沒有過關

### 🎯 任務 B5-3　營收排行

把 `revenue` 依金額**由高到低**印出排行（`第1名 珍珠奶茶 1,020 元`…），並印出 `總營收 3,690 元`。提示：`sorted(revenue.items(), key=lambda x: x[1], reverse=True)`。

In [ ]:
# 🎯 任務 B5-3　營收排行（請保留這一行）
revenue = {'珍珠奶茶': 1020, '雞排': 600, '滷肉飯': 990, '鹹酥雞': 480, '紅茶': 600}
ranking = sorted(revenue.items(), key=lambda x: x[1], reverse=True)
for i, (name, money) in enumerate(ranking, start=1):
    print(f"第{i}名 {name} {???} 元")
print(f"總營收 {???} 元")

In [ ]:
檢查("B5-3")   # ◀ 執行這一格，看看任務 B5-3 有沒有過關

### 🎯 任務 B5-4　營收柱狀圖

把 `revenue` 畫成柱狀圖：x 軸是品名、y 軸是營收，標題 `品項營收`，y 軸標籤 `營收（元）`，並存成 `revenue.png`。

In [ ]:
# 🎯 任務 B5-4　營收柱狀圖（請保留這一行）
import matplotlib.pyplot as plt
revenue = {'珍珠奶茶': 1020, '雞排': 600, '滷肉飯': 990, '鹹酥雞': 480, '紅茶': 600}
names = list(revenue.keys())
values = ???
plt.bar(names, values, color='teal')
plt.title(???)
plt.ylabel(???)
plt.savefig('revenue.png')
plt.show()

In [ ]:
檢查("B5-4")   # ◀ 執行這一格，看看任務 B5-4 有沒有過關

### 🎯 任務 B5-5　一鍵報表函式

把整個流程包成函式 `make_report(filename)`：讀檔（略過壞資料）→ 統計 → 印出排行與總營收 → 傳回 `revenue` 字典。呼叫 `make_report('sales.txt')`。

In [ ]:
# 🎯 任務 B5-5　一鍵報表函式（請保留這一行）
def make_report(filename):
    revenue = {}
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                name, qty, price = line.split(',')
                ???
            except ValueError:
                continue
    ranking = sorted(revenue.items(), key=lambda x: x[1], reverse=True)
    for i, (name, money) in enumerate(ranking, start=1):
        print(f"第{i}名 {name} {money:,} 元")
    print(f"總營收 {sum(revenue.values()):,} 元")
    return revenue
result = make_report('sales.txt')

In [ ]:
檢查("B5-5")   # ◀ 執行這一格，看看任務 B5-5 有沒有過關

## 🌟 進階挑戰（不計分）
1. 把排行結果寫成 `report.txt`。
2. 再畫一張**圓餅圖**顯示各品項營收占比。
3. 讓 `make_report()` 多一個參數 `top=3`，只印前 3 名。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🕷️ L12 網路爬蟲 requests + BeautifulSoup** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L12_web_scraping.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/